# DocMind RAG — Chunking Ablation Study
Compares 4 configurations:
- A: Fixed-size (400 words, no parent-child, no enrichment)
- B: Parent-child (800/150, no enrichment)
- C: Parent-child + contextual enrichment
- D: Parent-child + enrichment + tables atomic ← demo default

Expected: D > C > B > A. Tables-atomic gap should be the largest single jump.

In [ ]:
import json
import httpx

API_BASE = "http://localhost:8000/api/v1"

configs = {
    "A_fixed_size": {"parent_chunk_size": 400, "child_chunk_size": 400, "enrichment": False, "tables_atomic": False},
    "B_parent_child": {"parent_chunk_size": 800, "child_chunk_size": 150, "enrichment": False, "tables_atomic": False},
    "C_enriched": {"parent_chunk_size": 800, "child_chunk_size": 150, "enrichment": True, "tables_atomic": False},
    "D_full": {"parent_chunk_size": 800, "child_chunk_size": 150, "enrichment": True, "tables_atomic": True},
}

results = {}
for name, config in configs.items():
    print(f"Running config: {name}")
    r = httpx.post(f"{API_BASE}/eval/run", json={
        "dataset": "financebench",
        "sample_size": 50,
        "config": config,
    })
    results[name] = r.json()
    print(f"  Started: {results[name]}")

In [ ]:
# Compare results (run after all complete)
import time

for name, info in results.items():
    run_id = info.get("run_id") or info.get("task_id", "")
    for _ in range(60):
        r = httpx.get(f"{API_BASE}/eval/results/{run_id}")
        data = r.json()
        if data.get("status") in ("completed", "failed"):
            results[name] = data
            break
        time.sleep(10)

# Print comparison table
print(f"{'Config':<20} {'Hit Rate':<12} {'Faithful':<12} {'Relevancy':<12} {'Recall':<12}")
print("-" * 68)
for name, data in results.items():
    m = data.get("metrics", {})
    print(f"{name:<20} {m.get('retrieval_hit_rate', 0):<12.1%} {m.get('faithfulness', 0):<12.4f} {m.get('answer_relevancy', 0):<12.4f} {m.get('context_recall', 0):<12.4f}")